> **Cell note:** Introduces the notebook goal, topology, and validation flow.

# GRPO Rollout Path with vLLM + Weight Sync (2-GPU localhost)

This notebook demonstrates vLLM rollout with **end-to-end NCCL weight synchronization** on Red Hat OpenShift AI.

**Architecture:** vLLM gets 2 GPUs. It serves inference on GPU 0. A trainer script runs inside the same pod on GPU 1 and transfers weights over localhost via NCCL — matching the [official vLLM rlhf_http_nccl example](https://docs.vllm.ai/en/v0.21.0/examples/rl/rlhf_http_nccl/).

The flow:
1. Deploy vLLM with 2 GPUs and `--load-format dummy` (random weights → gibberish output)
2. Validate health, inference, and control-plane endpoints
3. Run a trainer script inside the vLLM pod that loads real weights on GPU 1 and syncs them to vLLM on GPU 0 via NCCL
4. Verify vLLM now produces correct output (gibberish → coherent answer)
5. **Prometheus metrics comparison** — scrape vLLM `/metrics` and run a latency/throughput benchmark before and after sync to quantify the impact


> **Cell note:** Installs Python packages required for cluster, API, and model operations.

> **Cell note:** Installs Python packages required for cluster, API, and model operations.

## Install dependencies


In [ ]:
# Cell note: Installs runtime dependencies used by later cells.
!python3 -m pip install -q -U kubeflow kubernetes openai requests transformers vllm


In [ ]:
# Cell note: Imports libraries for Kubernetes operations, HTTP checks, and inference calls.
import json
import os
import time

import requests
from kubernetes import client as k8s
from openai import OpenAI

> **Cell note:** Defines cluster auth, namespace/workbench detection, and rollout defaults.

## Configure cluster access and rollout service settings


In [ ]:
# Cell note: Resolves service-account auth/context and initializes Kubernetes/OpenShift clients.
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"
SA_NS_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"

if not os.getenv("OPENSHIFT_API_URL") and os.path.exists(SA_TOKEN_PATH):
    os.environ["OPENSHIFT_API_URL"] = "https://kubernetes.default.svc"
    with open(SA_TOKEN_PATH) as f:
        os.environ["NOTEBOOK_USER_TOKEN"] = f.read()
if not os.getenv("NOTEBOOK_NAMESPACE") and os.path.exists(SA_NS_PATH):
    with open(SA_NS_PATH) as f:
        os.environ["NOTEBOOK_NAMESPACE"] = f.read().strip()

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")
NAMESPACE = os.getenv("NOTEBOOK_NAMESPACE", "<your-namespace>")
VLLM_NAME = "grpo-vllm-rollout"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
IMAGE = os.getenv("VLLM_IMAGE", "vllm/vllm-openai:latest")
API_KEY = os.getenv("VLLM_API_KEY", "dummy-token")
RUN_OPTIONAL_VALIDATION = (
    os.getenv("RUN_OPTIONAL_VALIDATION", "true").lower() == "true"
)
WEIGHT_TRANSFER_BACKEND = os.getenv("VLLM_WEIGHT_TRANSFER_BACKEND", "nccl")
WEIGHT_SYNC_PORT = int(os.getenv("VLLM_WEIGHT_SYNC_PORT", "29501"))
ENABLE_FULL_WEIGHT_SYNC = (
    os.getenv("ENABLE_FULL_WEIGHT_SYNC", "true").lower() == "true"
)
VLLM_GPU_COUNT = int(os.getenv("VLLM_GPU_COUNT", "2"))
VLLM_GPU_MEM_UTIL = os.getenv("VLLM_GPU_MEM_UTIL", "0.4")
VLLM_LOAD_FORMAT = os.getenv("VLLM_LOAD_FORMAT", "dummy" if ENABLE_FULL_WEIGHT_SYNC else "auto")

if not api_server or not token:
    raise RuntimeError("OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN must be set.")
if NAMESPACE == "<your-namespace>":
    raise RuntimeError(
        "Set NOTEBOOK_NAMESPACE to your OpenShift AI project namespace before running this notebook."
    )

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = True
for candidate in [
    "/var/run/secrets/kubernetes.io/serviceaccount/ca.crt",
    os.getenv("SSL_CERT_FILE"),
    os.getenv("REQUESTS_CA_BUNDLE"),
]:
    if candidate and os.path.exists(candidate):
        configuration.ssl_ca_cert = candidate
        break
configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = k8s.ApiClient(configuration)
apps = k8s.AppsV1Api(api_client)
core = k8s.CoreV1Api(api_client)
networking = k8s.NetworkingV1Api(api_client)
workbench_pod_name = os.getenv("WORKBENCH_POD_NAME") or os.getenv("HOSTNAME")
workbench_name = (
    os.getenv("WORKBENCH_NAME")
    or os.getenv("NOTEBOOK_NAME")
    or (
        "-".join(workbench_pod_name.split("-")[:-1])
        if workbench_pod_name and "-" in workbench_pod_name
        else workbench_pod_name
    )
)
service_host = f"{VLLM_NAME}.{NAMESPACE}.svc.cluster.local"
service_root = f"http://{service_host}:8000"
base_url = f"{service_root}/v1"
health_url = f"{service_root}/health"
if not workbench_pod_name:
    raise RuntimeError(
        "WORKBENCH_POD_NAME or HOSTNAME must be set to identify the trainer-side workbench pod."
    )
workbench_pod = core.read_namespaced_pod(name=workbench_pod_name, namespace=NAMESPACE)
workbench_pod_ip = workbench_pod.status.pod_ip
print(f"Using rollout service host: {service_host}")
print(f"Optional validation enabled: {RUN_OPTIONAL_VALIDATION}")
print(f"Weight transfer backend: {WEIGHT_TRANSFER_BACKEND}")
print(f"Weight sync port: {WEIGHT_SYNC_PORT}")
print(f"Workbench pod: {workbench_pod_name} ({workbench_pod_ip})")
print(f"Workbench selector label: notebook-name={workbench_name}")
print(f"Full weight sync enabled: {ENABLE_FULL_WEIGHT_SYNC}")
print(f"vLLM GPUs: {VLLM_GPU_COUNT}, mem-util: {VLLM_GPU_MEM_UTIL}, load-format: {VLLM_LOAD_FORMAT}")

> **Cell note:** Introduces creation of vLLM rollout Deployment and Service resources.

## Create the vLLM rollout Deployment and Service


In [ ]:
# Cell note: Builds and applies rollout Deployment/Service specs with cache and runtime settings.
cache_dir = "/tmp/hf-cache"
home_dir = "/tmp/vllm-home"
flashinfer_dir = "/tmp/flashinfer-cache"

deployment = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {"name": VLLM_NAME, "namespace": NAMESPACE},
    "spec": {
        "replicas": 1,
        "selector": {"matchLabels": {"app": VLLM_NAME}},
        "template": {
            "metadata": {"labels": {"app": VLLM_NAME}},
            "spec": {
                "containers": [
                    {
                        "name": "vllm",
                        "image": IMAGE,
                        "args": [
                            MODEL_ID,
                            "--host",
                            "0.0.0.0",
                            "--port",
                            "8000",
                            "--gpu-memory-utilization",
                            VLLM_GPU_MEM_UTIL,
                            "--max-model-len",
                            "4096",
                            "--enforce-eager",
                            "--load-format",
                            VLLM_LOAD_FORMAT,
                            "--weight-transfer-config",
                            json.dumps({"backend": WEIGHT_TRANSFER_BACKEND}),
                        ],
                        "ports": [{"containerPort": 8000, "name": "http"}],
                        "env": [
                            {"name": "HF_TOKEN", "value": os.getenv("HF_TOKEN", "")},
                            {"name": "HOME", "value": home_dir},
                            {"name": "HF_HOME", "value": cache_dir},
                            {"name": "HUGGINGFACE_HUB_CACHE", "value": cache_dir},
                            {"name": "TRANSFORMERS_CACHE", "value": cache_dir},
                            {"name": "XDG_CACHE_HOME", "value": "/tmp"},
                            {"name": "VLLM_CACHE_ROOT", "value": "/tmp/vllm-cache"},
                            {"name": "VLLM_CONFIG_ROOT", "value": "/tmp/vllm-config"},
                            {
                                "name": "FLASHINFER_WORKSPACE_DIR",
                                "value": flashinfer_dir,
                            },
                            {"name": "VLLM_SERVER_DEV_MODE", "value": "1"},
                            {"name": "VLLM_ALLOW_INSECURE_SERIALIZATION", "value": "1"},
                        ],
                        "resources": {
                            "requests": {
                                "cpu": "4",
                                "memory": "24Gi",
                                "nvidia.com/gpu": str(VLLM_GPU_COUNT),
                            },
                            "limits": {
                                "cpu": "4",
                                "memory": "24Gi",
                                "nvidia.com/gpu": str(VLLM_GPU_COUNT),
                            },
                        },
                        "volumeMounts": [
                            {"name": "hf-cache", "mountPath": cache_dir},
                            {"name": "vllm-home", "mountPath": home_dir},
                            {"name": "flashinfer-cache", "mountPath": flashinfer_dir},
                        ],
                    }
                ],
                "volumes": [
                    {"name": "hf-cache", "emptyDir": {}},
                    {"name": "vllm-home", "emptyDir": {}},
                    {"name": "flashinfer-cache", "emptyDir": {}},
                ],
            },
        },
    },
}
service = {
    "apiVersion": "v1",
    "kind": "Service",
    "metadata": {"name": VLLM_NAME, "namespace": NAMESPACE},
    "spec": {
        "selector": {"app": VLLM_NAME},
        "ports": [{"port": 8000, "targetPort": 8000, "name": "http"}],
    },
}

try:
    apps.create_namespaced_deployment(namespace=NAMESPACE, body=deployment)
except Exception:
    apps.patch_namespaced_deployment(
        name=VLLM_NAME, namespace=NAMESPACE, body=deployment
    )

try:
    core.create_namespaced_service(namespace=NAMESPACE, body=service)
except Exception:
    core.patch_namespaced_service(name=VLLM_NAME, namespace=NAMESPACE, body=service)

print(f"Created or updated Deployment/Service for {VLLM_NAME}")

> **Cell note:** Explains the NetworkPolicy requirement for native weight-sync connectivity.

## Allow rollout-to-workbench traffic for native weight sync

Native weight sync requires the `vLLM` rollout pod to connect back to the GPU workbench on a fixed trainer rendezvous port. This cell creates a narrow NetworkPolicy to allow only that traffic.


In [ ]:
# Cell note: Creates/applies NetworkPolicy allowing rollout-to-workbench rendezvous traffic.
network_policy_name = f"{workbench_name}-vllm-weight-sync"
network_policy = {
    "apiVersion": "networking.k8s.io/v1",
    "kind": "NetworkPolicy",
    "metadata": {"name": network_policy_name, "namespace": NAMESPACE},
    "spec": {
        "podSelector": {"matchLabels": {"notebook-name": workbench_name}},
        "policyTypes": ["Ingress"],
        "ingress": [
            {
                "from": [{"podSelector": {"matchLabels": {"app": VLLM_NAME}}}],
                "ports": [{"protocol": "TCP", "port": WEIGHT_SYNC_PORT}],
            }
        ],
    },
}
try:
    networking.create_namespaced_network_policy(
        namespace=NAMESPACE, body=network_policy
    )
except Exception:
    networking.patch_namespaced_network_policy(
        name=network_policy_name, namespace=NAMESPACE, body=network_policy
    )
print(
    f"Created or updated NetworkPolicy {network_policy_name} for TCP/{WEIGHT_SYNC_PORT}"
)

> **Cell note:** Introduces readiness checks and API validation for rollout service.

## Wait for vLLM and validate the API


In [ ]:
# Cell note: Waits for rollout health and verifies key API endpoints respond correctly.
for _ in range(60):
    try:
        response = requests.get(health_url, timeout=5)
        if response.status_code == 200:
            print("vLLM health check passed")
            break
    except Exception:
        pass
    time.sleep(10)
else:
    raise RuntimeError("vLLM service did not become healthy in time")

> **Cell note:** Explains optional lightweight validation checks before full sync test.

## Optional native weight-sync validation

This section is disabled by default. Enable it only from a GPU-enabled workbench when you explicitly want to validate native weight transfer after the rollout path is already working. It is not required for the standard customer-facing rollout example.


In [ ]:
# Cell note: Runs optional control-plane and inference checks when the flag is enabled.
if not RUN_OPTIONAL_VALIDATION:
    print(
        "Skipping optional rollout API and control-plane validation. Set RUN_OPTIONAL_VALIDATION=true to execute these checks."
    )
else:
    client = OpenAI(base_url=base_url, api_key=API_KEY)
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": "Solve: If Sarah has 3 apples and gets 2 more, how many apples does she have? End with #### <number>.",
            }
        ],
        max_tokens=64,
    )
    print(response.choices[0].message.content)
    world_size_response = requests.get(f"{service_root}/get_world_size", timeout=10)
    world_size_response.raise_for_status()
    print("vLLM world size:", world_size_response.json())
    pause_response = requests.post(f"{service_root}/pause", timeout=30)
    pause_response.raise_for_status()
    print("pause status:", pause_response.status_code)
    resume_response = requests.post(f"{service_root}/resume", timeout=30)
    resume_response.raise_for_status()
    print("resume status:", resume_response.status_code)

> **Cell note:** Introduces full localhost 2-GPU weight-sync validation sequence.

## Full weight-sync validation (2-GPU localhost pattern)

This runs a trainer script **inside** the vLLM pod on GPU 1 while vLLM serves on GPU 0.
The NCCL rendezvous happens over localhost — matching the
[official vLLM example](https://docs.vllm.ai/en/v0.21.0/examples/rl/rlhf_http_nccl/).

With `--load-format dummy`, vLLM starts with random weights (gibberish output).
After weight sync, the model produces a correct answer — proving the transfer worked.


In [ ]:
# Cell note: Executes full in-pod trainer sync script and captures structured results.
if not ENABLE_FULL_WEIGHT_SYNC:
    print(
        "Skipping full weight sync. Set ENABLE_FULL_WEIGHT_SYNC=true to run."
    )
else:
    import subprocess
    import textwrap

    vllm_pods = core.list_namespaced_pod(
        namespace=NAMESPACE, label_selector=f"app={VLLM_NAME}"
    )
    vllm_pod_name = vllm_pods.items[0].metadata.name
    print(f"Target vLLM pod: {vllm_pod_name}")

    trainer_script = textwrap.dedent(f"""\
    import json as _json
    import threading, time, requests, torch
    from openai import OpenAI
    from transformers import AutoModelForCausalLM
    from vllm.distributed.weight_transfer.nccl_engine import (
        NCCLTrainerSendWeightsArgs, NCCLWeightTransferEngine,
    )
    from vllm.utils.network_utils import get_ip, get_open_port

    BASE_URL = "http://localhost:8000"
    MODEL = "{MODEL_ID}"
    METRICS_URL = f"{{BASE_URL}}/metrics"
    BENCHMARK_N = 20
    BENCHMARK_PROMPTS = [
        "What is 2+2? Answer with just the number.",
        "Name three primary colors.",
        "What planet is closest to the sun?",
        "Translate 'hello' to French.",
        "What is the square root of 144?",
    ]

    def scrape_metrics():
        raw = requests.get(METRICS_URL, timeout=10).text
        metrics = {{}}
        for line in raw.splitlines():
            if line.startswith("#"):
                continue
            for key in [
                "vllm:avg_generation_throughput_toks_per_s",
                "vllm:avg_prompt_throughput_toks_per_s",
                "vllm:num_requests_running",
                "vllm:num_requests_waiting",
                "vllm:gpu_cache_usage_perc",
                "vllm:cpu_cache_usage_perc",
            ]:
                if line.startswith(key):
                    try:
                        metrics[key] = float(line.split()[-1])
                    except ValueError:
                        pass
        return metrics

    def run_benchmark(label, n=BENCHMARK_N):
        client = OpenAI(base_url=f"{{BASE_URL}}/v1", api_key="dummy")
        latencies = []
        tok_counts = []
        for i in range(n):
            prompt = BENCHMARK_PROMPTS[i % len(BENCHMARK_PROMPTS)]
            t0 = time.perf_counter()
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{{"role": "user", "content": prompt}}],
                max_tokens=32,
            )
            elapsed = time.perf_counter() - t0
            latencies.append(elapsed)
            tok_counts.append(r.usage.completion_tokens)
        avg_lat = sum(latencies) / len(latencies)
        p50 = sorted(latencies)[len(latencies) // 2]
        p99 = sorted(latencies)[int(len(latencies) * 0.99)]
        total_toks = sum(tok_counts)
        total_time = sum(latencies)
        tps = total_toks / total_time if total_time > 0 else 0
        result = dict(
            label=label, requests=n,
            avg_latency_s=round(avg_lat, 4),
            p50_latency_s=round(p50, 4),
            p99_latency_s=round(p99, 4),
            total_tokens=total_toks,
            tokens_per_sec=round(tps, 2),
        )
        print(f"\\n[{{label}}] Benchmark ({{n}} requests):")
        for k, v in result.items():
            if k != "label":
                print(f"  {{k}}: {{v}}")
        return result

    ws = requests.get(f"{{BASE_URL}}/get_world_size", timeout=10).json()["world_size"]
    world_size = ws + 1
    device = f"cuda:{{ws}}"
    torch.cuda.set_device(int(device.split(":")[1]))
    print(f"Trainer on {{device}}, vLLM world_size={{ws}}, total={{world_size}}")

    print(f"Loading model: {{MODEL}}")
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to(device)

    client = OpenAI(base_url=f"{{BASE_URL}}/v1", api_key="dummy")
    print("=" * 60)
    print("BEFORE weight sync (dummy weights):")
    print("=" * 60)
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{{"role": "user", "content": "What is 2+2? Answer with just the number."}}],
        max_tokens=16,
    )
    print(f"  Answer: {{r.choices[0].message.content}}")
    pre_metrics = scrape_metrics()
    pre_bench = run_benchmark("BEFORE")

    addr = get_ip()
    port = get_open_port()
    print(f"\\nNCCL init: master={{addr}}:{{port}} world_size={{world_size}}")

    init_thread = threading.Thread(
        target=lambda: requests.post(
            f"{{BASE_URL}}/init_weight_transfer_engine",
            json={{"init_info": dict(master_address=addr, master_port=port, rank_offset=1, world_size=world_size)}},
            timeout=120,
        ).raise_for_status(),
    )
    init_thread.start()

    sync_t0 = time.perf_counter()
    group = NCCLWeightTransferEngine.trainer_init(
        dict(master_address=addr, master_port=port, world_size=world_size)
    )
    init_thread.join()
    print("NCCL group established")

    requests.post(f"{{BASE_URL}}/pause", timeout=30).raise_for_status()
    print("vLLM paused")

    requests.post(f"{{BASE_URL}}/start_weight_update", json={{"is_checkpoint_format": True}}, timeout=60).raise_for_status()

    names, dtypes, shapes = [], [], []
    for n, p in model.named_parameters():
        names.append(n)
        dtypes.append(str(p.dtype).split(".")[-1])
        shapes.append(list(p.shape))

    ut = threading.Thread(
        target=lambda: requests.post(
            f"{{BASE_URL}}/update_weights",
            json={{"update_info": dict(names=names, dtype_names=dtypes, shapes=shapes, packed=True)}},
            timeout=300,
        ).raise_for_status(),
    )
    ut.start()
    print("Broadcasting weights via NCCL...")
    NCCLWeightTransferEngine.trainer_send_weights(
        iterator=model.named_parameters(),
        trainer_args=NCCLTrainerSendWeightsArgs(group=group, packed=True),
    )
    ut.join()
    print("Weights transferred")

    requests.post(f"{{BASE_URL}}/finish_weight_update", json={{}}, timeout=60).raise_for_status()
    requests.post(f"{{BASE_URL}}/resume", timeout=30).raise_for_status()
    sync_elapsed = time.perf_counter() - sync_t0
    print(f"vLLM resumed — weight sync took {{sync_elapsed:.2f}}s")

    print("\\n" + "=" * 60)
    print("AFTER weight sync (real weights):")
    print("=" * 60)
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{{"role": "user", "content": "What is 2+2? Answer with just the number."}}],
        max_tokens=16,
    )
    print(f"  Answer: {{r.choices[0].message.content}}")
    post_metrics = scrape_metrics()
    post_bench = run_benchmark("AFTER")

    print("\\n" + "=" * 60)
    print("PROMETHEUS METRICS COMPARISON")
    print("=" * 60)
    all_keys = sorted(set(list(pre_metrics.keys()) + list(post_metrics.keys())))
    print(f"  {{'Metric':<52}} {{'Before':>10}} {{'After':>10}}")
    print(f"  {{'-'*52}} {{'-'*10}} {{'-'*10}}")
    for k in all_keys:
        pre_v = pre_metrics.get(k, "n/a")
        post_v = post_metrics.get(k, "n/a")
        if isinstance(pre_v, float):
            pre_v = f"{{pre_v:.4f}}"
        if isinstance(post_v, float):
            post_v = f"{{post_v:.4f}}"
        print(f"  {{k:<52}} {{str(pre_v):>10}} {{str(post_v):>10}}")

    print("\\n" + "=" * 60)
    print("LATENCY & THROUGHPUT COMPARISON")
    print("=" * 60)
    print(f"  {{'Metric':<30}} {{'Before':>12}} {{'After':>12}} {{'Delta':>12}}")
    print(f"  {{'-'*30}} {{'-'*12}} {{'-'*12}} {{'-'*12}}")
    for field in ["avg_latency_s", "p50_latency_s", "p99_latency_s", "tokens_per_sec", "total_tokens"]:
        bv = pre_bench[field]
        av = post_bench[field]
        delta = av - bv
        sign = "+" if delta >= 0 else ""
        print(f"  {{field:<30}} {{str(bv):>12}} {{str(av):>12}} {{sign}}{{delta:>11.4f}}")

    print(f"\\n  Weight sync duration: {{sync_elapsed:.2f}}s")
    print("\\n=== WEIGHT SYNC + METRICS COMPLETE ===")

    results = _json.dumps(dict(
        pre_metrics=pre_metrics, post_metrics=post_metrics,
        pre_bench=pre_bench, post_bench=post_bench,
        sync_duration_s=round(sync_elapsed, 2),
    ))
    print(f"\\n__RESULTS_JSON__{{results}}")
    """)

    script_path = "/tmp/weight_sync_test.py"
    with open(script_path, "w") as f:
        f.write(trainer_script)

    subprocess.run(
        ["oc", "cp", script_path, f"{NAMESPACE}/{vllm_pod_name}:{script_path}"],
        check=True,
    )
    print(f"Copied trainer script to {vllm_pod_name}:{script_path}")

    subprocess.run(
        ["oc", "exec", vllm_pod_name, "-n", NAMESPACE, "--",
         "pip", "install", "-q", "openai"],
        check=True,
    )

    print("\n--- Running weight sync inside vLLM pod ---\n")
    result = subprocess.run(
        ["oc", "exec", vllm_pod_name, "-n", NAMESPACE, "--",
         "bash", "-c", f"CUDA_VISIBLE_DEVICES=0,1 NCCL_DEBUG=WARN python3 {script_path}"],
        capture_output=True, text=True, timeout=600,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:] if result.stderr else "")
        raise RuntimeError(f"Weight sync failed with exit code {result.returncode}")
    print("Weight sync validated successfully from notebook!")

> **Cell note:** Introduces rendering of benchmark and metrics summary from sync output.

## Prometheus Metrics Summary

Parse the JSON blob from the weight sync script and render a comparison table.

In [ ]:
# Cell note: Parses JSON results and displays final comparison/metrics table.
if ENABLE_FULL_WEIGHT_SYNC and "result" in dir() and result.stdout:
    marker = "__RESULTS_JSON__"
    results_data = None
    for line in result.stdout.splitlines():
        if marker in line:
            results_data = json.loads(line.split(marker, 1)[1])
            break

    if results_data:
        from IPython.display import HTML, display

        pre_b = results_data["pre_bench"]
        post_b = results_data["post_bench"]
        pre_m = results_data["pre_metrics"]
        post_m = results_data["post_metrics"]
        sync_s = results_data["sync_duration_s"]

        def delta_cell(before, after, fmt=".4f", lower_better=True):
            d = after - before
            sign = "+" if d >= 0 else ""
            color = ""
            if lower_better:
                color = "green" if d < 0 else ("red" if d > 0 else "")
            else:
                color = "green" if d > 0 else ("red" if d < 0 else "")
            style = f' style="color:{color};font-weight:bold"' if color else ""
            return f"<td{style}>{sign}{d:{fmt}}</td>"

        rows = ""
        bench_fields = [
            ("Avg Latency (s)", "avg_latency_s", True, ".4f"),
            ("P50 Latency (s)", "p50_latency_s", True, ".4f"),
            ("P99 Latency (s)", "p99_latency_s", True, ".4f"),
            ("Tokens/sec", "tokens_per_sec", False, ".2f"),
            ("Total Tokens", "total_tokens", False, ".0f"),
        ]
        for label, key, lower_better, fmt in bench_fields:
            bv, av = pre_b[key], post_b[key]
            rows += f"<tr><td>{label}</td><td>{bv:{fmt}}</td><td>{av:{fmt}}</td>{delta_cell(bv, av, fmt, lower_better)}</tr>\n"

        prom_rows = ""
        for k in sorted(set(list(pre_m.keys()) + list(post_m.keys()))):
            short = k.replace("vllm:", "")
            bv = pre_m.get(k, 0)
            av = post_m.get(k, 0)
            prom_rows += f"<tr><td>{short}</td><td>{bv:.4f}</td><td>{av:.4f}</td><td>{av - bv:+.4f}</td></tr>\n"

        html = f"""
        <h3>Benchmark: Before vs After Weight Sync ({pre_b['requests']} requests)</h3>
        <table border="1" cellpadding="6" cellspacing="0" style="border-collapse:collapse; font-family:monospace;">
        <tr style="background:#f0f0f0"><th>Metric</th><th>Before (dummy)</th><th>After (real)</th><th>Delta</th></tr>
        {rows}
        <tr style="background:#f8f8f8"><td colspan="4"><b>Weight sync duration: {sync_s}s</b></td></tr>
        </table>
        <br/>
        <h3>Prometheus vLLM Metrics</h3>
        <table border="1" cellpadding="6" cellspacing="0" style="border-collapse:collapse; font-family:monospace;">
        <tr style="background:#f0f0f0"><th>Metric</th><th>Before</th><th>After</th><th>Delta</th></tr>
        {prom_rows}
        </table>
        """
        display(HTML(html))
    else:
        print("Could not parse metrics JSON from weight sync output.")
else:
    print("Skipped — weight sync was not run.")